In [1]:
# 1. 최신 transformers (SigLIP2용, Jina와 반대)
!pip install -q "transformers>=4.50" peft accelerate
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
"""
SigLIP2 성능 평가 (LookMatch) — Jina-CLIP과 비교용
===================================================
SigLIP2의 파인튜닝 전후 Recall@K를 측정.
Jina-CLIP evaluate와 같은 방식(검증셋 = 학습 미사용분)으로 공정 비교.

RUN_EXP2: 파인튜닝 전후 (원본 SigLIP2 vs 파인튜닝 SigLIP2)

입력:  metadata_captioned_final.csv
       siglip2_finetuned/ (파인튜닝 결과)

실행:  python evaluate_siglip2.py
"""

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm


def setup_data_dir():
    env_dir = os.environ.get("LOOKMATCH_DIR")
    if env_dir:
        return Path(env_dir)
    try:
        import google.colab  # noqa
        drive_root = Path("/content/drive/MyDrive")
        if not drive_root.exists():
            from google.colab import drive
            drive.mount("/content/drive")
        return drive_root / "clothes_dataset"
    except ImportError:
        return Path("clothes_dataset")


DATA_DIR = setup_data_dir()
IN_CSV = DATA_DIR / "metadata_captioned_final.csv"
if not IN_CSV.exists():
    for alt in ["metadata_captioned.csv"]:
        if (DATA_DIR / alt).exists():
            IN_CSV = DATA_DIR / alt
            break

CROPS_SUBDIR = "crops"
LOCAL_CROPS = Path("/content/crops_local")

MODEL_NAME = "google/siglip2-so400m-patch16-512"   # finetune과 동일하게
FINETUNED_DIR = str(DATA_DIR / "siglip2_finetuned_full")

N_EVAL = None            # 검증셋 중 N개 (전체는 None)
K_LIST = [1, 5, 10]
BATCH = 32
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def resolve_image_path(crop_path):
    fname = str(crop_path).replace("\\", "/").split("/")[-1]
    for cand in [LOCAL_CROPS / fname, Path(crop_path), DATA_DIR / CROPS_SUBDIR / fname]:
        if Path(cand).exists():
            return str(cand)
    return None


def load_eval_data():
    """finetune과 동일 분할 → 검증셋(학습 미사용)만 평가"""
    df = pd.read_csv(IN_CSV)
    df = df[df["qwen_caption"].notna() &
            (df["qwen_caption"].astype(str).str.strip() != "")]
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    n_val = max(1, int(len(df) * 0.1))
    val_df = df[:n_val]

    rows = []
    for _, r in val_df.iterrows():
        img = resolve_image_path(r["crop_path"])
        naver = str(r.get("caption", "")).strip()
        qwen = str(r.get("qwen_caption", "")).strip()
        if img and qwen:
            rows.append({"image": img, "naver": naver, "qwen": qwen})
    random.Random(42).shuffle(rows)
    if N_EVAL and len(rows) > N_EVAL:
        rows = rows[:N_EVAL]
    print(f"평가 샘플(학습 미사용 검증셋): {len(rows)}개")
    return rows


_PROC = {}
def _get_processor():
    if "p" not in _PROC:
        from transformers import AutoProcessor
        _PROC["p"] = AutoProcessor.from_pretrained(MODEL_NAME)
    return _PROC["p"]


def load_model(finetuned_dir=None):
    from transformers import AutoModel
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    if finetuned_dir and Path(finetuned_dir).exists():
        try:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, finetuned_dir).to(DEVICE)
            print(f"  파인튜닝 로드: {finetuned_dir}")
        except Exception as e:
            print(f"  [경고] 파인튜닝 로드 실패({e}) → 원본")
    model.eval()
    return model


def get_base(model):
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        return model.base_model.model
    return model


def _to_tensor(out):
    """get_*_features 반환이 텐서면 그대로, 객체면 임베딩 텐서 추출"""
    if torch.is_tensor(out):
        return out
    # 출력 객체면 흔한 임베딩 속성 순서대로 시도
    for attr in ["image_embeds", "text_embeds", "pooler_output", "last_hidden_state"]:
        if hasattr(out, attr):
            v = getattr(out, attr)
            if torch.is_tensor(v):
                # last_hidden_state면 평균 풀링
                if attr == "last_hidden_state" and v.dim() == 3:
                    return v.mean(dim=1)
                return v
    raise TypeError(f"임베딩 텐서를 찾을 수 없음: {type(out)}")


@torch.no_grad()
def embed_all(model, rows, caption_key="qwen"):
    base = get_base(model)
    proc = _get_processor()
    img_embs, txt_embs = [], []
    for i in tqdm(range(0, len(rows), BATCH), desc=f"임베딩({caption_key})", leave=False):
        chunk = rows[i:i+BATCH]
        images = [Image.open(r["image"]).convert("RGB") for r in chunk]
        texts = [r[caption_key] for r in chunk]
        img_in = proc(images=images, return_tensors="pt").to(DEVICE)
        txt_in = proc(text=texts, return_tensors="pt", padding="max_length",
                      max_length=64, truncation=True).to(DEVICE)
        img_out = _to_tensor(base.get_image_features(**img_in))
        txt_out = _to_tensor(base.get_text_features(**txt_in))
        ie = F.normalize(img_out.float(), dim=-1)
        te = F.normalize(txt_out.float(), dim=-1)
        img_embs.append(ie.cpu()); txt_embs.append(te.cpu())
    return torch.cat(img_embs), torch.cat(txt_embs)


def recall_at_k(txt_emb, img_emb, k_list):
    sims = txt_emb @ img_emb.t()
    n = sims.size(0)
    results = {}
    for k in k_list:
        topk = sims.topk(min(k, sims.size(1)), dim=1).indices
        correct = sum(i in topk[i] for i in range(n))
        results[k] = correct / n
    return results


def print_recall(title, r):
    print(f"  {title}: " + " / ".join(f"R@{k}={v:.3f}" for k, v in r.items()))


RUN_EXP1 = False          # 캡션 스타일 비교 (SigLIP2 한국어 성능 확인)
RUN_EXP2 = True          # 파인튜닝 전후

def main():
    print(f"입력: {IN_CSV}")
    print(f"모델: {MODEL_NAME}")
    if not IN_CSV.exists():
        print("[오류] CSV 없음"); return

    rows = load_eval_data()

    # ── 실험 1: 캡션 스타일 비교 (원본, 한국어 성능 확인) ──
    if RUN_EXP1:
        print("\n" + "="*60)
        print("[실험 1] 캡션 스타일 비교 (원본 SigLIP2, 한국어 검색 성능)")
        print("  네이버 제목 vs Qwen 캡션 — SigLIP2가 한국어를 이해하는가")
        print("="*60)
        model = load_model()
        naver_rows = [r for r in rows if r["naver"]]
        print(f"\n네이버 제목 있는 샘플: {len(naver_rows)}개")

        i_n, t_n = embed_all(model, naver_rows, "naver")
        r_naver = recall_at_k(t_n, i_n, K_LIST)
        i_q, t_q = embed_all(model, naver_rows, "qwen")
        r_qwen = recall_at_k(t_q, i_q, K_LIST)

        print("\n결과 (텍스트→이미지):")
        print_recall("네이버 제목으로 검색", r_naver)
        print_recall("Qwen 캡션으로 검색  ", r_qwen)
        print("\n해석: 값이 낮으면 SigLIP2의 한국어 검색이 약한 것")
        print("      → Jina-CLIP(한국어 강)과 비교 근거")
        del model
        torch.cuda.empty_cache() if DEVICE=="cuda" else None

    # ── 실험 2: 파인튜닝 전후 ──
    if RUN_EXP2:
        print("\n" + "="*60)
        print("[실험 2] 파인튜닝 전후 비교")
        print("="*60)

        print("\n원본 SigLIP2:")
        m0 = load_model()
        i0, t0 = embed_all(m0, rows, "qwen")
        r0 = recall_at_k(t0, i0, K_LIST)
        del m0; torch.cuda.empty_cache() if DEVICE=="cuda" else None

        print("파인튜닝 SigLIP2:")
        m1 = load_model(FINETUNED_DIR)
        i1, t1 = embed_all(m1, rows, "qwen")
        r1 = recall_at_k(t1, i1, K_LIST)

        print("\n결과 (텍스트→이미지):")
        print_recall("파인튜닝 전(원본)", r0)
        print_recall("파인튜닝 후      ", r1)
        print("\n개선폭:")
        for k in K_LIST:
            print(f"  R@{k}: {r0[k]:.3f} → {r1[k]:.3f} ({r1[k]-r0[k]:+.3f})")

    print("\n" + "="*60)
    print("평가 완료")
    print("="*60)


if __name__ == "__main__":
    main()


입력: /content/drive/MyDrive/clothes_dataset/metadata_captioned_final.csv
모델: google/siglip2-so400m-patch16-512
평가 샘플(학습 미사용 검증셋): 7163개

[실험 2] 파인튜닝 전후 비교

원본 SigLIP2:


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

파인튜닝 SigLIP2:


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

  파인튜닝 로드: /content/drive/MyDrive/clothes_dataset/siglip2_finetuned_full



결과 (텍스트→이미지):
  파인튜닝 전(원본): R@1=0.108 / R@5=0.256 / R@10=0.351
  파인튜닝 후      : R@1=0.285 / R@5=0.585 / R@10=0.723

개선폭:
  R@1: 0.108 → 0.285 (+0.176)
  R@5: 0.256 → 0.585 (+0.329)
  R@10: 0.351 → 0.723 (+0.372)

평가 완료
